In [1]:
import numpy as np
from matplotlib import pyplot as plt
import numba
get_ipython().run_line_magic('matplotlib', 'auto')

Using matplotlib backend: MacOSX


In [30]:
def init(xmax):
    plt.xlim((0, xmax-1))
    plt.ylim(-1,1)
    plt.grid('on')
    ax.set_xlabel('Grid Cells ($z$)')
    ax.set_ylabel('$E_x$')
    plt.show()

Everything here is in natural units

# 1D

## No axion, sourceless AED

In [ ]:
def update1d(Ex, By):
    for x in range(1,kmax-1): # only update kmax -2 many cells
        Ex[x] = Ex[x] + 0.5 * (By[x-1] - By[x])
    for x in range(0, kmax-1):
        By[x] = By[x] + 0.5 * (Ex[x] - Ex[x+1])
    return Ex, By


In [ ]:
kmax = 160
nsteps = 100
Ex = np.zeros(kmax, float)
By = np.zeros(kmax, float)
xrange = np.linspace(0,kmax, kmax)

for i in range(0,nsteps+1):
    plt.clf()
    Ex, By = update1d(Ex, By)
    plt.plot(xrange, Ex)
    plt.show()

## No axion, soft source AED

plot source

In [ ]:
def get_source(t):
    t0 = nsteps/5
    spread = nsteps/30
    source = np.exp(-0.5*(t-t0)**2/spread**2)
    return source
plt.clf()
plt.close()
plt.plot(np.arange(nsteps), get_source(np.arange(nsteps)))
plt.xlim(0,nsteps)
plt.ylim(-1,1)
plt.show()

new update equation with source

In [ ]:
def update1d(Ex, By, source):
    for x in range(1,kmax-1): # only update kmax -2 many cells
        Ex[x] = Ex[x] + 0.5 * (By[x-1] - By[x])
    # source injectin
    Ex[sourceidx] =  Ez[sourceidx] - source*0.5
    for x in range(0, kmax-1):
        By[x] = By[x] + 0.5 * (Ex[x] - Ex[x+1])
    return Ex, By

In [ ]:
kmax = 800
nsteps = 2000
Ex = np.zeros(kmax, float)
By = np.zeros(kmax, float)
xrange = np.linspace(0,kmax, kmax)

# source
sourceidx = int(kmax/4)

plt.clf()
cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ex,linewidth=lw)
#[im2] = ax.plot(xrange,By,linewidth=lw)
init(kmax)


for i in range(0,nsteps+1):
    
    source = get_source(i)
    Ex, By = update1d(Ex, By, source)
    
    if i % cycle == 0:
        im.set_ydata(Ex)
        #im2.set_ydata(By)
        ax.set_title("frame time {}".format(i))
        plt.show()
        plt.pause(0.01)
        
print('done')

## No axion, pulse source (ABC, TFSF)

plot source

In [17]:
nsteps = 2000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6

def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*np.pi*0.01)
    return source

#plt.xlim(0,20)
plt.clf()
plt.plot(t, get_source(t))
plt.show()
print(t0, spread)

360 60


update equation with pulse

In [18]:
def Eupdate1d(Ex, By, source):
    for x in range(1,kmax-1): # only update kmax -2 many cells
        Ex[x] = Ex[x] + 0.5 * (By[x-1] - By[x])
    # source injectin
    Ex[sourceidx] = Ex[sourceidx] - source*0.5
    return Ex

def Bupdate1d(Ex, By, source):
    for x in range(0, kmax-1):
        By[x] = By[x] + 0.5 * (Ex[x] - Ex[x+1])
    By[sourceidx-1] = By[sourceidx-1] - source*0.5
    return By

In [19]:
nsteps = 2000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6


kmax = 800
nsteps = 2000
Ex = np.zeros(kmax, float)
By = np.zeros(kmax, float)
xrange = np.linspace(0,kmax, kmax)

# source
sourceidx = int(kmax/4)

plt.clf()
plt.close()
cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ex,linewidth=lw)
init(kmax)

def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*np.pi*0.01)
    return source

for i in range(0,nsteps+1):
    
    t = i-1
    source = get_source(i)
    source2 = get_source(i+0.5)
    
    Eleft_0 = Ex[1] # for 2 timesteps later
    Eright_0 = Ex[-2]
    
    Ex = Eupdate1d(Ex, By, source2)
    
    # ABC
    if i == 0:
        Eleft_ = None
        Eright_ = None
    
    if i != 0:    
        Ex[0] = Eleft_
        Ex[-1] = Eright_
    
    Eleft_ = Eleft_0
    Eright_ = Eright_0
        
    By = Bupdate1d(Ex, By, source)
    
    
    
    if i % cycle == 0:
        im.set_ydata(Ex)
        #plt.plot(xrange, Ez)
        ax.set_title("frame time {}".format(i))
        plt.show()
        plt.pause(0.01)
print('done')

done


## No axion, plane wave AED (ABC, TFSF)

In [ ]:
def get_source(t):
    E0 = 0.5
    wavelength=800
    t0 = 0
    #w = 2*np.pi/(wavelength)
    w = 0.01*np.pi
    #mask = (t>t0)*1
    source = E0*np.sin(w*t)#*mask
    return source

plt.close()
spread = 60
t0 = spread*6
nsteps=2000
plt.plot(np.arange(0,nsteps), get_source(np.arange(0,nsteps)), label='source')
plt.ylim(-1,1)
plt.legend()
plt.show()

In [26]:
def Eupdate1d(Ex, By, source):
    for x in range(1,kmax-1): # only update kmax -2 many cells
        Ex[x] = Ex[x] + 0.5 * (By[x-1] - By[x])
    # source injectin
    Ex[sourceidx] = Ex[sourceidx] - source*0.5
    return Ex

def Bupdate1d(Ex, By, source):
    for x in range(0, kmax-1):
        By[x] = By[x] + 0.5 * (Ex[x] - Ex[x+1])
    By[sourceidx-1] = By[sourceidx-1] - source*0.5 
    return By

In [ ]:
"spacegrid has to be 10 times smaller than the wavelength"

nsteps = 2000
t = np.arange(0,nsteps+1)
wavelength = 800 # in gridspace
t0 = 100

kmax = 1000
nsteps = 2000
Ex = np.zeros(kmax, float)
By = np.zeros(kmax, float)
xrange = np.linspace(0,kmax, kmax)

# source
sourceidx = int(kmax/4)

cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ex,linewidth=lw)
#[im2] = ax.plot(xrange,By,linewidth=lw)
init(kmax)

def get_source(t):
    E0 = 0.5
    #w = 2*np.pi/(wavelength)
    w = 0.005*np.pi
    source = E0*np.sin(w*t)
    return source

for i in range(0,nsteps+1):
    
    source = get_source(i)
    source2 = get_source(i+0.5)
    
    Eleft_0 = Ex[1] # for 2 timesteps later
    Eright_0 = Ex[-2]
    
    Ex = Eupdate1d(Ex, By, source2)
    
    # ABC
    if i == 0:
        Eleft_ = None
        Eright_ = None
    
    if i != 0:    
        Ex[0] = Eleft_
        Ex[-1] = Eright_
    
    Eleft_ = Eleft_0
    Eright_ = Eright_0
        
    By = Bupdate1d(Ex, By, source)
    
    
    
    if i % cycle == 0:
        im.set_ydata(Ex)
        #im2.set_ydata(By)
        #plt.plot(xrange, Ez)
        ax.set_title("frame time {}".format(i))
        plt.show()
        plt.pause(0.01)
print('done')

# I found that there is no way to implement AED with 1D, the code below is the wrong 1D AED, thus it's only for reference

## sourceless AED with uniform background B field (Bx) 

In [27]:
def axionUpdate1d(Ex,By,axion, axion_past):
    # need global variable m, dt
    for x in range(1,kmax-1): # only update kmax -2 many cells
        Ex[x] = Ex[x] + 0.5 * (By[x] - By[x-1])
    for x in range(0, kmax-1):
        By[x] = By[x] + 0.5 * (Ex[x+1] - Ex[x])
    for x in range(1, kmax-1): # only update kmax -2 many cells
        axion[x] = 2* axion[x] - axion_past[x] + 0.5*(axion[x+1] - 2*axion[x]+axion[x-1]) \
        - dt**2 *(axion[x]*By[x]**2/(1+axion[x]**2)**2) + dt**2 *(axion[x]*Ez[x]**2/(1+axion[x]**2)**2)\
        - dt**2 * m**2 * axion[x]
    return Ex, By, axion

In [ ]:
kmax = 160
nsteps = 100
Ex = np.zeros(kmax, float)
By = np.zeros(kmax, float)
axion = np.zeros(kmax, float)
xrange = np.linspace(0, kmax, kmax)
dx = 1
dt = 0.5**0.5 * dx
m = 1 
for i in range(0, nsteps+1):
    if i == 0:
        axion_past = np.zeros(kmax, float)
    Ex, By, axion = axionUpdate1d(Ex, By, axion, axion_past)
    if i == 0:
        axion_current = np.zeros(kmax, float)
    axion_past = axion_current
    axion_current = axion
    

Ex_phy = Ex/(1+axion**2)
Ey_phy = By*axion/(1+axion**2)
plt.plot(xrange, Ex_phy)
plt.plot(xrange, Ey_phy, '--')

## soft source AED

plot source

In [ ]:
def get_source(t):
    t0 = 2000/5
    spread = 2000/30
    source = np.exp(-0.5*(t-t0)**2/spread**2)
    return source
nsteps = 2000
plt.clf()
plt.plot(np.arange(nsteps), get_source(np.arange(nsteps)))
plt.xlim(0,nsteps)
plt.ylim(-1,1)
plt.show()

In [ ]:
# update equation with source injection
def axionUpdate1d(Ez,By,axion, axion_past, source):
    # need global variable m, dt
    for x in range(1,kmax-1): # only update kmax -2 many cells
        Ez[x] = Ez[x] + 0.5 * (By[x] - By[x-1])
        
    # convert Ehat to physical field
    Ey_phy, Ez_phy, By_phy, Bz_phy = hat2phy(Ez, By, axion)
    # inject source
    Ez_phy[sourceidx] = Ez_phy[sourceidx] - source*0.5
    # convert physical field back to Ehat
    Ey, Ez, By, Bz = phy2hat(Ey_phy, Ez_phy, By_phy, Bz_phy, axion)    
    
    for x in range(0, kmax-1):
        By[x] = By[x] + 0.5 * (Ez[x+1] - Ez[x])
    for x in range(1, kmax-1): # only update kmax -2 many cells
        axion[x] = 2* axion[x] - axion_past[x] + 0.5*(axion[x+1] - 2*axion[x]+axion[x-1]) \
        - dt**2 *(axion[x]*By[x]**2/(1+axion[x]**2)**2) + dt**2 *(axion[x]*Ez[x]**2/(1+axion[x]**2)**2)\
        - dt**2 * m**2 * axion[x]
    return Ez, By, axion
        

injecting a pulse to the physical E field

In [ ]:
kmax = 800
nsteps = 4000
Ez = np.zeros(kmax, float)
By = np.zeros(kmax, float)

Ey = np.zeros(kmax, float)
Bz = np.zeros(kmax, float)

Ez_phy = np.zeros(kmax, float)
By_phy = np.zeros(kmax, float)

Ey_phy = np.zeros(kmax, float)
Bz_phy = np.zeros(kmax, float)

axion = np.zeros(kmax, float)
xrange = np.linspace(0, kmax, kmax)
dx = 1
dt = 0.5**0.5 * dx
m = 1 

# source
sourceidx = int(kmax/4)

plt.close()
cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ez,linewidth=lw)
[im2] = ax.plot(xrange,axion,linewidth=lw)
init(kmax)


def hat2phy(Ez, By, axion):
    Ey_phy = axion * By/(1+axion**2)
    Ez_phy = Ez/(1+axion**2)
    By_phy = By/(1+axion**2)
    Bz_phy = axion*Ez/(1+axion**2)
    
    return Ey_phy, Ez_phy, By_phy, Bz_phy

def phy2hat(Ey_phy, Ez_phy, By_phy, Bz_phy, axion):
    Ez = Ez_phy - axion*Bz_phy
    By = By_phy + axion*Ey_phy
#     Ey = np.zeros_like(Ez)
#     Bz = np.zeros_like(By)
    
    return Ey, Ez, By, Bz
    

for i in range(0, nsteps+1):
    source = get_source(i)
    if i == 0:
        axion_past = np.zeros(kmax, float)
    Ez, By, axion = axionUpdate1d(Ez, By, axion, axion_past, source)
    
    if i == 0:
        axion_current = np.zeros(kmax, float)
    axion_past = axion_current
    axion_current = axion
    
    if i % cycle == 0:
        Ey_phy, Ez_phy, By_phy, Bz_phy = hat2phy(Ez, By, axion)
        im.set_ydata(Ez_phy)
        im2.set_ydata(Ey)
        ax.set_title("frame time {}".format(i))
        plt.show()
        plt.pause(0.01)

print('done')

## Pulse AED

plot source

In [7]:
nsteps = 2000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6

def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*0.05)
    return source

#plt.xlim(0,20)
plt.clf()
plt.plot(t, get_source(t))
plt.xlim(0,nsteps)
plt.ylim(-1,1)
plt.show()
print(t0, spread)

360 60


In [ ]:
# update equation with source injection
def axionUpdate1d(Ez,By,axion, axion_past, source):
    # need global variable m, dt
    for x in range(1,kmax-1): # only update kmax -2 many cells
        Ez[x] = Ez[x] + 0.5 * (By[x] - By[x-1])
        
    # convert Ehat to physical field
    Ey_phy, Ez_phy, By_phy, Bz_phy = hat2phy(Ez, By, axion)
    # inject source
    Ez_phy[sourceidx] = Ez_phy[sourceidx] - source*0.5
    # convert physical field back to Ehat
    Ey, Ez, By, Bz = phy2hat(Ey_phy, Ez_phy, By_phy, Bz_phy, axion)    
    
    for x in range(0, kmax-1):
        By[x] = By[x] + 0.5 * (Ez[x+1] - Ez[x])
    for x in range(1, kmax-1): # only update kmax -2 many cells
        axion[x] = 2* axion[x] - axion_past[x] + 0.5*(axion[x+1] - 2*axion[x]+axion[x-1]) \
        - dt**2 *(axion[x]*By[x]**2/(1+axion[x]**2)**2) + dt**2 *(axion[x]*Ez[x]**2/(1+axion[x]**2)**2)\
        - dt**2 * m**2 * axion[x]
    return Ez, By, axion
        

In [ ]:
kmax = 800
nsteps = 4000
Ez = np.zeros(kmax, float)
By = np.zeros(kmax, float)

Ey = np.zeros(kmax, float)
Bz = np.zeros(kmax, float)

Ez_phy = np.zeros(kmax, float)
By_phy = np.zeros(kmax, float)

Ey_phy = np.zeros(kmax, float)
Bz_phy = np.zeros(kmax, float)

axion = np.zeros(kmax, float)
xrange = np.linspace(0, kmax, kmax)
dx = 1
dt = 0.5**0.5 * dx
m = 1 

# source
sourceidx = int(kmax/4)

plt.close()
cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ez,linewidth=lw)
[im2] = ax.plot(xrange,axion,linewidth=lw)
init(kmax)


def hat2phy(Ez, By, axion):
    Ey_phy = axion * By/(1+axion**2)
    Ez_phy = Ez/(1+axion**2)
    By_phy = By/(1+axion**2)
    Bz_phy = axion*Ez/(1+axion**2)
    
    return Ey_phy, Ez_phy, By_phy, Bz_phy

def phy2hat(Ey_phy, Ez_phy, By_phy, Bz_phy, axion):
    Ez = Ez_phy - axion*Bz_phy
    By = By_phy + axion*Ey_phy
#     Ey = np.zeros_like(Ez)
#     Bz = np.zeros_like(By)
    
    return Ey, Ez, By, Bz
    

for i in range(0, nsteps+1):
    source = get_source(i)
    if i == 0:
        axion_past = np.zeros(kmax, float)
    Ez, By, axion = axionUpdate1d(Ez, By, axion, axion_past, source)
    
    if i == 0:
        axion_current = np.zeros(kmax, float)
    axion_past = axion_current
    axion_current = axion
    
    if i % cycle == 0:
        Ey_phy, Ez_phy, By_phy, Bz_phy = hat2phy(Ez, By, axion)
        im.set_ydata(Ez_phy)
        im2.set_ydata(Ey)
        ax.set_title("frame time {}".format(i))
        plt.show()
        plt.pause(0.01)

print('done')